# Agentes de Voz en Tiempo Real

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/voz-audio/03-agentes-voz-realtime.ipynb)

⚠️ **Nota**: Las celdas de captura de micrófono requieren ejecución local (no funcionan en Colab). El pipeline de texto sí funciona en Colab.

Construcción de agentes de voz con VAD, pipeline STT→LLM→TTS y OpenAI Realtime API.

In [ ]:
!pip install openai anthropic -q
# Para uso local con micrófono:
# pip install webrtcvad pyaudio

In [ ]:
import openai
import anthropic
import io
import wave
import time

openai_client = openai.OpenAI()
anthropic_client = anthropic.Anthropic()

## 1. Pipeline STT → Claude → TTS (sin micrófono)

In [ ]:
import re

def pipeline_texto_a_audio(texto_usuario: str, historial: list, voz: str = 'nova') -> tuple:
    """
    Simula el pipeline de voz usando texto en lugar de micrófono.
    En producción, sustituir texto_usuario por la salida de Whisper.
    """
    t0 = time.time()
    
    # 1. Claude genera respuesta
    historial.append({'role': 'user', 'content': texto_usuario})
    response = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=150,
        system='Asistente vocal: responde en máximo 2 frases, sin markdown.',
        messages=historial,
    )
    texto_resp = re.sub(r'[*#`]', '', response.content[0].text).strip()
    historial.append({'role': 'assistant', 'content': texto_resp})
    t_llm = time.time() - t0
    
    # 2. TTS genera audio
    audio = openai_client.audio.speech.create(model='tts-1', voice=voz, input=texto_resp)
    t_total = time.time() - t0
    
    return texto_resp, audio.content, historial, {'llm_ms': int(t_llm*1000), 'total_ms': int(t_total*1000)}

# Demo de conversación
historial = []
preguntas = [
    '¿Qué temperatura hace en Madrid hoy?',
    '¿Y para mañana?',
    '¿Necesito paraguas?',
]

for pregunta in preguntas:
    texto, audio, historial, latencias = pipeline_texto_a_audio(pregunta, historial)
    print(f'👤 {pregunta}')
    print(f'🤖 {texto}')
    print(f'⏱  LLM: {latencias["llm_ms"]}ms | Total: {latencias["total_ms"]}ms\n')

## 2. Benchmark de latencia del pipeline

In [ ]:
import statistics

def benchmark_pipeline(n_iter: int = 5) -> dict:
    """Mide la latencia del pipeline LLM + TTS."""
    preguntas_test = [
        '¿Cuál es la capital de Francia?',
        '¿Qué es el machine learning?',
        'Dame un consejo de productividad.',
        '¿Cuánto es 15% de 200?',
        'Explica qué es un LLM en una frase.',
    ]
    
    tiempos_llm = []
    tiempos_tts = []
    
    for i in range(n_iter):
        pregunta = preguntas_test[i % len(preguntas_test)]
        
        t0 = time.time()
        response = anthropic_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=100,
            messages=[{'role': 'user', 'content': pregunta}],
        )
        t_llm = time.time() - t0
        tiempos_llm.append(t_llm)
        
        texto = response.content[0].text[:200]
        t1 = time.time()
        openai_client.audio.speech.create(model='tts-1', voice='nova', input=texto)
        t_tts = time.time() - t1
        tiempos_tts.append(t_tts)
        
        print(f'  Iteración {i+1}: LLM={t_llm*1000:.0f}ms TTS={t_tts*1000:.0f}ms Total={(t_llm+t_tts)*1000:.0f}ms')
    
    return {
        'llm_p50_ms': int(statistics.median(tiempos_llm) * 1000),
        'tts_p50_ms': int(statistics.median(tiempos_tts) * 1000),
        'total_p50_ms': int((statistics.median(tiempos_llm) + statistics.median(tiempos_tts)) * 1000),
    }

print('Benchmarking pipeline voz...')
resultados = benchmark_pipeline(5)
print(f'\n=== Resultados (mediana) ===')
print(f'LLM (Claude Haiku): {resultados["llm_p50_ms"]}ms')
print(f'TTS (OpenAI tts-1): {resultados["tts_p50_ms"]}ms')
print(f'Total pipeline: {resultados["total_p50_ms"]}ms')
print(f'\n⚡ Objetivo para experiencia natural: < 800ms')
print(f'Estado: {"✅ OK" if resultados["total_p50_ms"] < 800 else "⚠️ Lento"}')

## 3. Esquema de la OpenAI Realtime API

In [ ]:
# Esquema de conexión a la Realtime API (requiere ejecución local con websockets)
REALTIME_SCHEMA = {
    'endpoint': 'wss://api.openai.com/v1/realtime?model=gpt-4o-realtime-preview-2024-10-01',
    'headers': {
        'Authorization': 'Bearer {OPENAI_API_KEY}',
        'OpenAI-Beta': 'realtime=v1',
    },
    'session_config': {
        'modalities': ['text', 'audio'],
        'voice': 'alloy',
        'input_audio_format': 'pcm16',
        'output_audio_format': 'pcm16',
        'sample_rate': 24000,
        'turn_detection': {
            'type': 'server_vad',
            'threshold': 0.5,
            'silence_duration_ms': 600,
        },
    },
    'ventajas': [
        'Latencia total ~300ms (vs ~800ms pipeline tradicional)',
        'VAD gestionado por OpenAI (sin webrtcvad local)',
        'Interrupción natural del modelo cuando el usuario habla',
        'Soporte para herramientas (function calling) en tiempo real',
    ],
    'limitaciones': [
        'Solo disponible con modelos GPT-4o (no Claude)',
        'Requiere WebSocket persistente (no REST)',
        'Coste más alto que pipeline tradicional',
        'Audio a 24kHz PCM16 (requiere conversión desde micrófono estándar)',
    ],
}

import json
print('=== OpenAI Realtime API — Resumen ===')
print(f'Endpoint: {REALTIME_SCHEMA["endpoint"]}')
print(f'\nVentajas:')
for v in REALTIME_SCHEMA['ventajas']:
    print(f'  ✅ {v}')
print(f'\nLimitaciones:')
for l in REALTIME_SCHEMA['limitaciones']:
    print(f'  ⚠️  {l}')